# 01. Signal Validation (Indian Markets Pivot)
This notebook demonstrates the underlying logic of the Physical Activity Index (PAI) and how it correlates with the Revenue Surprise target for the Indian retail universe.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('dark_background')
sns.set_palette('flare')

config = {
    'mode': 'synthetic',          # 'real' | 'synthetic'
    'market': 'NSE_INDIA',
    'universe': ['DMART', 'TRENT', 'JUBLFOOD', 'WESTLIFE', 'DEVYANI', 'SHOPERSTOP', 'VMART', 'ABFRL', 'SPENCERS', 'SAPPHIRE', 'BARBEQUE', 'METRO'],
    'date_range': ('2019-01-01', '2024-12-31'),
    'data_sources': {
        'satellite': 'sentinel2_gee',
        'earnings':  'nse_filings',
        'estimates': 'trendlyne'
    }
}
print("=== SPLM INDIA SATELLITE PIPELINE ===")
for k, v in config.items(): print(f"{k}: {v}")

### Step 1: Generate Synthetic Data
Since we are bypassing the GEE/Sentinel download in this presentation, we generate synthetic data that mathematically matches what the pipeline expects.

In [ ]:
# Synthetic Data Generation
dates = pd.date_range(start=config['date_range'][0], end=config['date_range'][1], freq='QE')
records = []

for t in config['universe']:
    true_alpha = np.random.normal(0, 0.1)
    for d in dates:
        is_monsoon = d.quarter == 3
        # Physical signal leading the financial signal
        pai = np.random.normal(true_alpha, 0.5) if not is_monsoon else np.nan
        rev_surp = pai * 0.3 + np.random.normal(0, 0.8) if pd.notna(pai) else np.random.normal(0, 1.0)
        
        records.append({
            'ticker': t,
            'date': d,
            'year': d.year,
            'quarter': d.quarter,
            'pai_zscore': pai,
            'revenue_surprise_zscore': rev_surp,
            'sector': 'qsr' if t in ['JUBLFOOD', 'WESTLIFE', 'DEVYANI', 'SAPPHIRE'] else 'retail'
        })

df = pd.DataFrame(records)
df['delta_signal_smooth'] = df['pai_zscore'] - df.groupby('ticker')['revenue_surprise_zscore'].shift(1).fillna(0)

print("Generated Synthetic PAI and Revenue Surprise Targets.")
display(df.tail())

### Step 2: Information Coefficient (IC) Analysis
How well does `delta_signal` predict the N-quarter forward revision?

In [ ]:
import scipy.stats as stats

# Calculate Forward Targets
for w in [1, 2, 3, 4]:
    df[f'fwd_rev_{w}q'] = df.groupby('ticker')['revenue_surprise_zscore'].shift(-w)

ic_results = []
for w in [1, 2, 3, 4]:
    t_col = f'fwd_rev_{w}q'
    def calc_spearman(group):
        valid = ~(group['delta_signal_smooth'].isna() | group[t_col].isna())
        if valid.sum() < 5: return np.nan
        return stats.spearmanr(group['delta_signal_smooth'][valid], group[t_col][valid])[0]
        
    quarterly_ic = df.groupby(['year', 'quarter']).apply(calc_spearman).dropna()
    ic_results.append({
        "Horizon": f"Q+{w}",
        "IC Mean": quarterly_ic.mean(),
        "IC Std": quarterly_ic.std(),
        "IR": quarterly_ic.mean() / quarterly_ic.std(),
        "Hit Rate": (quarterly_ic > 0).mean()
    })

ic_df = pd.DataFrame(ic_results)
display(ic_df)

### Step 3: PAI Time Series Visualization
Let's look at a specific ticker, such as DMART, to visualize physical congestion.

In [ ]:
dmart = df[df['ticker'] == 'DMART'].set_index('date')

plt.figure(figsize=(12, 6))
plt.plot(dmart.index, dmart['pai_zscore'], marker='o', label='Physical Activity Index (PAI)', color='#ff9933')
plt.plot(dmart.index, dmart['revenue_surprise_zscore'], linestyle='--', label='Reported Rev Surprise', color='#79c0ff')
plt.axhline(0, color='gray', linewidth=0.5)
plt.title("DMART: Physical Congestion vs Revenue Delivery")
plt.xlabel("Date")
plt.ylabel("Z-Score")
plt.legend()
plt.tight_layout()
plt.show()